In [1]:
import pandas as pd
import glob
import plotly.express as px
import numpy as np
from plotly.subplots import make_subplots
from plotly.graph_objects import Figure

# Data Importation and Concatination

In [2]:
datasets_location = '/content/drive/MyDrive/Nairobi Air Quality Data/*.csv'

all_files = glob.glob(datasets_location)

df_list = [pd.read_csv(file, on_bad_lines="skip") for file in all_files]

df = pd.concat(df_list, ignore_index=True)

print(all_files)

['/content/drive/MyDrive/Nairobi Air Quality Data/january 2026.csv', '/content/drive/MyDrive/Nairobi Air Quality Data/february 2026.csv', '/content/drive/MyDrive/Nairobi Air Quality Data/April 2026.csv', '/content/drive/MyDrive/Nairobi Air Quality Data/march 2026.csv', '/content/drive/MyDrive/Nairobi Air Quality Data/May 2026.csv']


In [3]:
# parse standard ISO8601 strings
df["timestamp"] = pd.to_datetime(df["timestamp"], format="ISO8601")

In [4]:
df = df.sort_values(by="timestamp",ascending=True).reset_index(drop=True)

In [35]:
df.head()

,location,timestamp,P0,P1,P2,humidity,temperature
0,3981,2026-01-01 00:00:00+00:00,18.816667,33.263333,29.610000,88.584615,20.069231
1,3981,2026-01-01 01:00:00+00:00,20.816667,37.900000,33.200000,90.425000,19.212500
2,3981,2026-01-01 02:00:00+00:00,21.911111,39.584444,33.986111,91.700000,19.057143
3,3981,2026-01-01 03:00:00+00:00,20.258333,35.230556,31.475000,90.752941,18.723529
4,3981,2026-01-01 04:00:00+00:00,16.942857,28.950000,27.253571,91.078571,19.092857


In [6]:
df.shape

(504927, 9)

In [7]:
df.columns

Index(['sensor_id', 'sensor_type', 'location', 'lat', 'lon', 'timestamp',
       'value_type', 'value', 'Unnamed: 8'],
      dtype='object')

In [8]:
print(df['location'].value_counts())

location
3981    231193
76      204793
3967     43187
3966     25743
34          11
Name: count, dtype: int64


In [9]:
df.drop("Unnamed: 8", axis=1, inplace=True)
df.head()

,sensor_id,sensor_type,location,lat,lon,timestamp,value_type,value
0,4977,DHT22,76,-1.261,36.782,2026-01-01 00:00:00+00:00,humidity,93.1
1,4977,DHT22,76,-1.261,36.782,2026-01-01 00:00:00+00:00,temperature,18.7
2,4898,pms5003,3966,-1.311,36.817,2026-01-01 00:02:43.459736+00:00,P2,28
3,4898,pms5003,3966,-1.311,36.817,2026-01-01 00:02:43.459736+00:00,P1,30
4,4898,pms5003,3966,-1.311,36.817,2026-01-01 00:02:43.459736+00:00,P0,20


In [10]:
df['sensor_type'].unique()

array(['DHT22', 'pms5003'], dtype=object)

# Sensor Type Explanation

1. **DHT22** – measures temperature and relative humidity. Typical range: -40°C to 80°C (±0.5°C accuracy) and 0–100% RH (±2% RH accuracy). It communicates over a single-wire digital interface.

2. **PMS5003** – a laser-based particulate matter (PM) sensor that measures airborne particles such as PM1.0, PM2.5, and PM10 using a UART serial interface. It operates from a 5 V supply and outputs particle concentration data.

In [11]:
df['value_type'].unique()

array(['humidity', 'temperature', 'P2', 'P1', 'P0'], dtype=object)

In [12]:
df.groupby('sensor_type')['value_type'].value_counts()

sensor_type  value_type 
DHT22        humidity        99320
             temperature     99319
             P0                103
             P1                103
             P2                103
pms5003      P2             101585
             P1             101584
             P0             101582
             humidity          614
             temperature       614
Name: count, dtype: int64

In [13]:
# mapping dictionary
location_map = {
    34: "Loresho (Residential Baseline)",
    76: "Westlands / Parklands (Commercial Hub)",
    3966: "Nairobi CBD East (Downtown area)",
    3967: "Nairobi CBD West (Haile Selassie/Moi Avenue Corridor)",
    3981: "Pangani / Muthaiga Border (Urban Transition)",
}

# Column Additon
df["neighborhood"] = df["location"].map(location_map)

In [14]:
df.head()

,sensor_id,sensor_type,location,lat,lon,timestamp,value_type,value,neighborhood
0,4977,DHT22,76,-1.261,36.782,2026-01-01 00:00:00+00:00,humidity,93.1,Westlands / Parklands (Commercial Hub)
1,4977,DHT22,76,-1.261,36.782,2026-01-01 00:00:00+00:00,temperature,18.7,Westlands / Parklands (Commercial Hub)
2,4898,pms5003,3966,-1.311,36.817,2026-01-01 00:02:43.459736+00:00,P2,28,Nairobi CBD East (Downtown area)
3,4898,pms5003,3966,-1.311,36.817,2026-01-01 00:02:43.459736+00:00,P1,30,Nairobi CBD East (Downtown area)
4,4898,pms5003,3966,-1.311,36.817,2026-01-01 00:02:43.459736+00:00,P0,20,Nairobi CBD East (Downtown area)


In [15]:
df.dtypes

,0
sensor_id,int64
sensor_type,object
location,int64
lat,float64
lon,float64
timestamp,"datetime64[ns, UTC]"
value_type,object
value,object
neighborhood,object


In [16]:
df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Pivot table

In [17]:
# Pivot using the numeric location column
df_pivot = df.pivot_table(
    index=["timestamp", "location"],
    columns="value_type",
    values="value",
    aggfunc="mean",
)

# I Resampled to hourly steps per location group
df_continuous = (
    df_pivot.groupby("location").resample("h", level="timestamp").mean()
)

# Flatten back out into normal columns
df_continuous = df_continuous.reset_index().dropna()

df_continuous.head()

value_type,location,timestamp,P0,P1,P2,humidity,temperature
0,34,2026-02-27 11:00:00+00:00,53.333333,82.466667,67.333333,32.700000,35.400000
1,76,2026-01-01 00:00:00+00:00,11.600000,22.400000,20.450000,93.338095,18.438095
2,76,2026-01-01 01:00:00+00:00,9.850000,19.500000,17.600000,93.555000,18.470000
3,76,2026-01-01 02:00:00+00:00,10.952381,21.952381,19.523810,94.304762,18.180952
4,76,2026-01-01 03:00:00+00:00,11.823529,23.176471,20.647059,95.070588,17.788235
5,76,2026-01-01 04:00:00+00:00,9.333333,20.333333,17.333333,92.666667,17.900000
6,76,2026-01-01 05:00:00+00:00,6.777778,15.111111,12.444444,84.688889,20.200000
7,76,2026-01-01 06:00:00+00:00,6.100000,13.500000,10.900000,70.480000,23.560000
8,76,2026-01-01 07:00:00+00:00,7.000000,12.600000,10.900000,57.690000,26.240000
9,76,2026-01-01 08:00:00+00:00,6.200000,11.400000,10.500000,51.740000,27.550000


In [18]:
df_continuous.columns.name = None


In [19]:
df_continuous.head()

,location,timestamp,P0,P1,P2,humidity,temperature
0,34,2026-02-27 11:00:00+00:00,53.333333,82.466667,67.333333,32.700000,35.400000
1,76,2026-01-01 00:00:00+00:00,11.600000,22.400000,20.450000,93.338095,18.438095
2,76,2026-01-01 01:00:00+00:00,9.850000,19.500000,17.600000,93.555000,18.470000
3,76,2026-01-01 02:00:00+00:00,10.952381,21.952381,19.523810,94.304762,18.180952
4,76,2026-01-01 03:00:00+00:00,11.823529,23.176471,20.647059,95.070588,17.788235
5,76,2026-01-01 04:00:00+00:00,9.333333,20.333333,17.333333,92.666667,17.900000
6,76,2026-01-01 05:00:00+00:00,6.777778,15.111111,12.444444,84.688889,20.200000
7,76,2026-01-01 06:00:00+00:00,6.100000,13.500000,10.900000,70.480000,23.560000
8,76,2026-01-01 07:00:00+00:00,7.000000,12.600000,10.900000,57.690000,26.240000
9,76,2026-01-01 08:00:00+00:00,6.200000,11.400000,10.500000,51.740000,27.550000


# Clipping to protect gradient descent

In [23]:
target_columns = ["P0", "P1", "P2"]

df_continuous[target_columns] = df_continuous[target_columns].clip(upper=350)

In [24]:
# Sort
df_continuous = df_continuous.sort_values(by=["location", "timestamp"]).reset_index(
    drop=True
)

In [25]:
df_continuous.shape

(7432, 7)

| Location ID | Location Name                 | Description                       | Land Use / Expected Air Quality Characteristics                                                                                                              |
| ----------: | ----------------------------- | --------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------ |
|      **34** | **Loresho**                   | Residential Baseline              | Low-density residential neighborhood used as a baseline/reference site with relatively low traffic emissions.                                                |
|      **76** | **Westlands / Parklands**     | Commercial Hub                    | Mixed commercial and residential district with busy roads, offices, restaurants, and moderate to heavy traffic.                                              |
|    **3966** | **Downtown Nairobi**          | Commercial & Public Transport Hub | Dense commercial district with matatu termini, heavy pedestrian activity, and persistent traffic congestion; expected to exhibit elevated PM concentrations. |
|    **3967** | **Nairobi CBD West**          | Haile Selassie Corridor           | Major transport corridor characterized by continuous vehicle flow, bus operations, and high commuter activity.                                               |
|    **3981** | **Pangani / Muthaiga Border** | Urban Transition Zone             | Transitional area between residential and commercial land use, influenced by commuter traffic but generally lower emissions than the CBD.                    |


In [26]:
df_westlands = df_continuous[df_continuous['location'] == 76].reset_index(drop=True)
df_downtown = df_continuous[df_continuous['location'] == 3966].reset_index(drop=True)
df_cbd_west = df_continuous[df_continuous['location'] == 3967].reset_index(drop=True)
df_muthaiga = df_continuous[df_continuous['location'] == 3981].reset_index(drop=True)

In [27]:
for df in [df_westlands, df_downtown, df_cbd_west, df_muthaiga]:
    df['timestamp'] = pd.to_datetime(df['timestamp'])

for df in [df_westlands, df_downtown, df_cbd_west, df_muthaiga]:
    df.sort_values(by='timestamp', inplace=True)
    df.reset_index(drop=True, inplace=True)

In [28]:
# This counts the total number of hourly rows assigned to each location
print(df_continuous['location'].value_counts())


location
3981    3011
76      2544
3967    1370
3966     506
34         1
Name: count, dtype: int64


In [29]:
# Keep only the robust sensors that actually have continuous data
df_continuous = df_continuous[df_continuous["location"] != 34].reset_index(
    drop=True
)

# Verify who is left
print("Active locations for training:")
print(df_continuous["location"].unique())

Active locations for training:
[  76 3966 3967 3981]


In [30]:
df_westlands.sort_values(by='timestamp', inplace=True)
# Strip the timezone data to make it a standard local datetime
# Updated title to reflect PM10 for P1
x='timestamp'

y='P2'
fig_westlands = px.line(df_westlands, x,y, title=f'Westlands {y} Concentration')
fig_westlands.show()

In [31]:
px.line(df_cbd_west,x,y,title=f"Nairobi CBD West {y} Concentration")

In [32]:
px.line(df_downtown,x,y,title=f"Downtown Nairobi {y} Concentration")

In [33]:
px.line(df_muthaiga,x,y,title=f"Pangani/Muthaiga {y} Concentration")

In [34]:
df_continuous.head()

,location,timestamp,P0,P1,P2,humidity,temperature
0,76,2026-01-01 00:00:00+00:00,11.600000,22.400000,20.450000,93.338095,18.438095
1,76,2026-01-01 01:00:00+00:00,9.850000,19.500000,17.600000,93.555000,18.470000
2,76,2026-01-01 02:00:00+00:00,10.952381,21.952381,19.523810,94.304762,18.180952
3,76,2026-01-01 03:00:00+00:00,11.823529,23.176471,20.647059,95.070588,17.788235
4,76,2026-01-01 04:00:00+00:00,9.333333,20.333333,17.333333,92.666667,17.900000


In [36]:
df_continuous.to_csv('Nairobi_Air_Quality_cleaned.csv', index=False)